In [1]:
# check the predictions and how mlflow does this
# pull these out, or generate them
# go to the scoring for defenders and work out what you can do here + what data you need
#https://www.premierleague.com/en/news/2174909/fpl-basics-scoring
# Features

Features to use:
- ~~predicted minutes~~
- goal record (player or team level?)
- assist record (player or team level?)
- clean sheets (player + team level?)
- defensive contributions (player + team level - also what does this mean?) [link]([company/ymiqeug7uo99i6fj8aom/asset-search-etl](https://www.premierleague.com/en/news/4361991/whats-new-in-202526-fantasy-defensive-contributions))
- disciplinary record (player + team)
- goals conceded (team level)
- home vs away 
- opposition strength

In [2]:
import duckdb
import polars as pl
from matplotlib import pyplot as plt

from fantasy_football.constants import DATA_FOLDER

In [3]:
DATABASE_PATH = DATA_FOLDER / "fantasy_football.duckdb"
connection = duckdb.connect(DATABASE_PATH)

In [15]:
res = connection.sql("SELECT * FROM player_week").pl()
res.head(10)

season,gw,element,name,position,team,bonus,minutes,round,total_points,value
str,i64,i64,str,str,str,i64,i64,i64,i64,i64
"""2024-25""",10,488,"""Fraser Forster""","""GK""","""Spurs""",0,0,10,0,44
"""2024-25""",18,651,"""Orel Mangala""","""MID""","""Everton""",0,90,18,1,50
"""2024-25""",34,499,"""Pape Matar Sarr""","""MID""","""Spurs""",0,45,34,1,47
"""2024-25""",3,411,"""Lewis Miley""","""MID""","""Newcastle""",0,0,3,0,50
"""2024-25""",6,247,"""Alex Iwobi""","""MID""","""Fulham""",0,90,6,3,55
"""2024-25""",4,285,"""Boubakary Soumaré""","""MID""","""Leicester""",0,0,4,0,45
"""2024-25""",18,342,"""Bernardo Veiga de Carvalho e S…","""MID""","""Man City""",2,90,18,9,63
"""2024-25""",4,469,"""Alex McCarthy""","""GK""","""Southampton""",0,0,4,0,45
"""2024-25""",14,142,"""Jason Steele""","""GK""","""Brighton""",0,0,14,0,42


In [ ]:
match = connection.sql("SELECT * FROM player_match").pl()
match.head(10)

season,gw,element,opponent,is_home,minutes,total_points,kickoff_time
str,i64,i64,i64,bool,i64,i64,datetime[μs]
"""2024-25""",1,77,16,false,62,2,2024-08-17 14:00:00
"""2024-25""",1,427,3,true,0,0,2024-08-17 14:00:00
"""2024-25""",1,22,20,true,0,0,2024-08-17 14:00:00
"""2024-25""",1,197,4,false,0,0,2024-08-18 13:00:00
"""2024-25""",1,584,15,false,70,1,2024-08-17 14:00:00
"""2024-25""",1,52,19,false,90,2,2024-08-17 16:30:00
"""2024-25""",1,215,4,false,0,0,2024-08-18 13:00:00
"""2024-25""",1,609,11,false,0,0,2024-08-19 19:00:00
"""2024-25""",1,550,1,false,90,2,2024-08-17 14:00:00


In [ ]:
minutes_prediction = connection.sql("SELECT * FROM minutes_prediction").pl()
minutes_prediction.head(10)

season,gw,element,opponent,p_zero,p_partial,p_sixty_plus,expected_minutes,model_version,prediction_kind,snapshot_captured_at
str,i64,i64,i64,f64,f64,f64,f64,str,str,datetime[μs]
"""2016-17""",1,6,9,0.180194,0.077763,0.742044,57.986142,"""7""","""backfill""",null
"""2016-17""",1,7,9,0.401603,0.077084,0.521312,41.410956,"""7""","""backfill""",null
"""2016-17""",1,11,9,0.528008,0.06779,0.404202,32.348861,"""7""","""backfill""",null
"""2016-17""",1,13,9,0.126715,0.171621,0.701664,57.773438,"""7""","""backfill""",null
"""2016-17""",1,14,9,0.044124,0.130769,0.825107,65.806122,"""7""","""backfill""",null
"""2016-17""",1,16,9,0.097705,0.16228,0.740015,60.369518,"""7""","""backfill""",null
"""2016-17""",1,17,9,0.272456,0.185248,0.542296,46.22967,"""7""","""backfill""",null
"""2016-17""",1,18,9,0.272456,0.185248,0.542296,46.22967,"""7""","""backfill""",null
"""2016-17""",1,21,9,0.272456,0.185248,0.542296,46.22967,"""7""","""backfill""",null


In [20]:
player_season = connection.sql("SELECT DISTINCT season FROM player_season").pl()
player_season.head(10)

season
str
"""2019-20"""
"""2017-18"""
"""2024-25"""
"""2025-26"""
"""2020-21"""
"""2018-19"""
"""2026-27"""
"""2021-22"""
"""2016-17"""


In [22]:
base_sql = """
SELECT 
    m.season
    ,s.first_name || ' ' || s.second_name as player_name
    ,m.element
    ,m.total_points
    ,m.kickoff_time
    ,m.gw
    ,m.opponent
    ,m.minutes
    ,mn.expected_minutes
    ,mn.p_zero
    ,mn.p_partial
    ,mn.p_sixty_plus

FROM player_match m 

LEFT JOIN player_season s ON m.element = s.element
    AND m.season = s.season
LEFT JOIN minutes_prediction mn ON m.element = mn.element
    AND m.season = mn.season
    AND m.gw = mn.gw
    AND m.element = mn.element
    AND m.opponent = mn.opponent

WHERE m.season = '2025-26'
"""

base = connection.sql(base_sql).pl()
base.head(10)


season,player_name,element,total_points,kickoff_time,gw,opponent,minutes,expected_minutes,p_zero,p_partial,p_sixty_plus
str,str,i64,i64,datetime[μs],i64,i64,i64,f64,f64,f64,f64
"""2025-26""","""Reinildo Mandava""",541,6,2025-08-16 14:00:00,1,19,90,39.220002,0.405154,0.119855,0.474991
"""2025-26""","""Lewis Dobbin""",57,0,2025-08-16 11:30:00,1,15,0,17.053931,0.67393,0.164474,0.161596
"""2025-26""","""Ryan Christie""",87,0,2025-08-15 19:00:00,1,12,0,0.701187,0.983624,0.011711,0.004665
"""2025-26""","""Zeki Amdouni""",216,0,2025-08-16 14:00:00,1,18,0,0.220596,0.994143,0.00486,0.000997
"""2025-26""","""Lucas Tolentino Coelho de Lima""",612,2,2025-08-16 14:00:00,1,17,90,55.443915,0.125518,0.225382,0.649099
"""2025-26""","""Jorrel Hato""",672,0,2025-08-17 13:00:00,1,8,0,47.238095,0.300674,0.115808,0.583518
"""2025-26""","""Myles Peart-Harris""",133,0,2025-08-17 13:00:00,1,16,0,18.052887,0.64928,0.183358,0.167362
"""2025-26""","""Ben Gannon-Doak""",391,0,2025-08-15 19:00:00,1,4,0,20.54283,0.612145,0.189918,0.197937
"""2025-26""","""Bashir Humphreys""",193,0,2025-08-16 14:00:00,1,18,0,1.071356,0.980764,0.008252,0.010984
